# Uplift Trees / Uplift Random Forest Experiments

Этот ноутбук проверяет идею Uplift Trees: дерево выбирает сплиты не по RMSE, а по тому, насколько в листе различаются treatment/control.

Что здесь делаем:

- переводим цель в бинарный факт покупки `rec_spend > 0`, потому что классические uplift-деревья обычно работают с binary outcome;
- учим Uplift Random Forest с критериями `kl`, `entropy/js`, `ed`;
- отдельно оцениваем размер чека через CatBoost-регрессор по положительным покупкам;
- итоговый score: `uplift_probability_of_purchase * expected_positive_spend`;
- сравниваем по `uplift@10` и `bootstrap lower 80 CI`;
- сохраняем несколько сабмитов-кандидатов.

Это не обязано победить CatBoost Hurdle T-learner в одиночку, но может дать другой ранжирующий сигнал для blend.

In [ ]:
import gc
import json
import os
import random
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

RUN_MODE = os.environ.get("UPLIFT_TREE_RUN_MODE", "smoke").lower()
assert RUN_MODE in {"smoke", "vm", "full"}

MODE_CFG = {
    "smoke": {
        "n_configs": 3,
        "n_estimators": 20,
        "max_depth": 4,
        "n_thresholds": 5,
        "bootstrap_iters": 80,
        "final_seeds": [42],
    },
    "vm": {
        "n_configs": 12,
        "n_estimators": 90,
        "max_depth": 5,
        "n_thresholds": 8,
        "bootstrap_iters": 220,
        "final_seeds": [42, 202, 777],
    },
    "full": {
        "n_configs": 22,
        "n_estimators": 150,
        "max_depth": 6,
        "n_thresholds": 10,
        "bootstrap_iters": 350,
        "final_seeds": [42, 202, 777, 1001, 3407],
    },
}
CFG = MODE_CFG[RUN_MODE]
print("RUN_MODE:", RUN_MODE)
print(json.dumps(CFG, indent=2))

DATA_DIR = Path("кей")
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"
ARTIFACT_DIR = Path("uplift_tree_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

ID_COL = "user_id"
TARGET_COL = "rec_spend"
TREATMENT_COL = "treatment_flg"
COMM_COL = "communication_type"
TOP_K = 0.10

## Load Data

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
FEATURES = [c for c in test.columns if c != ID_COL]

print("train:", train.shape)
print("test :", test.shape)
print("zero share:", float((train[TARGET_COL] == 0).mean()))
print("features:", len(FEATURES))

display(train.groupby([COMM_COL, TREATMENT_COL])[TARGET_COL].agg(["size", "mean", "median"]))

## Metric

In [ ]:
def uplift_at_k(y_true, treatment, score, k=TOP_K):
    y_true = np.asarray(y_true, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    score = np.asarray(score, dtype=float)
    n_top = max(1, int(len(score) * k))
    top_idx = np.argsort(score)[-n_top:]
    tr = treatment[top_idx] == 1
    ct = treatment[top_idx] == 0
    if tr.sum() == 0 or ct.sum() == 0:
        return np.nan
    return float(y_true[top_idx][tr].mean() - y_true[top_idx][ct].mean())


def bootstrap_uplift_ci(y_true, treatment, score, k=TOP_K, n_iter=200, ci=0.80, seed=RANDOM_STATE):
    y_true = np.asarray(y_true, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    score = np.asarray(score, dtype=float)
    rng = np.random.default_rng(seed)
    n_top = max(1, int(len(score) * k))
    top_idx = np.argsort(score)[-n_top:]
    vals = []
    for _ in range(n_iter):
        idx = rng.choice(top_idx, size=n_top, replace=True)
        tr = treatment[idx] == 1
        ct = treatment[idx] == 0
        if tr.sum() == 0 or ct.sum() == 0:
            continue
        vals.append(y_true[idx][tr].mean() - y_true[idx][ct].mean())
    vals = np.asarray(vals, dtype=float)
    alpha = (1 - ci) / 2
    return {
        "bootstrap_mean": float(vals.mean()) if len(vals) else np.nan,
        "bootstrap_std": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
        "lower80": float(np.quantile(vals, alpha)) if len(vals) else np.nan,
        "upper80": float(np.quantile(vals, 1 - alpha)) if len(vals) else np.nan,
        "iters_used": int(len(vals)),
    }


def evaluate_scores(df, score, label):
    point = uplift_at_k(df[TARGET_COL], df[TREATMENT_COL], score)
    ci = bootstrap_uplift_ci(df[TARGET_COL], df[TREATMENT_COL], score, n_iter=CFG["bootstrap_iters"])
    n_top = max(1, int(len(score) * TOP_K))
    top = df.iloc[np.argsort(np.asarray(score))[-n_top:]]
    row = {
        "model": label,
        "uplift_at10": point,
        "lower80": ci["lower80"],
        "upper80": ci["upper80"],
        "bootstrap_std": ci["bootstrap_std"],
        "top_target_mean": float(top[TARGET_COL].mean()),
        "top_treatment_share": float(top[TREATMENT_COL].mean()),
    }
    if COMM_COL in top.columns:
        for comm, share in top[COMM_COL].value_counts(normalize=True).sort_index().items():
            row[f"top_{comm}_share"] = float(share)
    return row


def objective(row):
    return 0.78 * row["lower80"] + 0.22 * row["uplift_at10"]


def rank_average(*scores, weights=None):
    arrs = [np.asarray(s, dtype=float) for s in scores]
    if weights is None:
        weights = np.ones(len(arrs), dtype=float) / len(arrs)
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    out = np.zeros_like(arrs[0], dtype=float)
    for s, w in zip(arrs, weights):
        out += w * pd.Series(s).rank(method="average").to_numpy()
    return out


def save_submission(scores, path):
    sub = pd.DataFrame({ID_COL: test[ID_COL].to_numpy(), "UPLIFT_SCORE": np.asarray(scores, dtype=float)})
    assert len(sub) == len(test)
    assert sub[ID_COL].equals(test[ID_COL].reset_index(drop=True))
    assert sub["UPLIFT_SCORE"].notna().all()
    sub.to_csv(path, index=False)
    print("saved", path, sub.shape)
    return sub

## Model Matrix For Uplift Trees

In [ ]:
def make_tree_matrix(train_df, test_df, features):
    all_df = pd.concat([train_df[features], test_df[features]], axis=0, ignore_index=True)
    encoded = []
    names = []
    for col in features:
        s_train = train_df[col]
        s_all = all_df[col]
        if pd.api.types.is_numeric_dtype(s_train):
            med = float(s_train.median()) if s_train.notna().any() else 0.0
            x_all = s_all.astype(float).fillna(med).to_numpy(dtype=np.float32)
            encoded.append(x_all)
            names.append(col)
            # Missingness flags often carry signal in this case.
            if s_all.isna().mean() >= 0.03:
                encoded.append(s_all.isna().astype(np.float32).to_numpy())
                names.append(f"{col}__isna")
        else:
            vals = s_all.astype("string").fillna("__MISSING__")
            codes, uniques = pd.factorize(vals, sort=True)
            encoded.append(codes.astype(np.float32))
            names.append(col)
    X_all = np.vstack(encoded).T.astype(np.float32)
    X_train = X_all[: len(train_df)]
    X_test = X_all[len(train_df) :]
    return X_train, X_test, names

X_tree_train, X_tree_test, TREE_FEATURE_NAMES = make_tree_matrix(train, test, FEATURES)
y_purchase = (train[TARGET_COL].to_numpy(dtype=float) > 0).astype(np.int8)
treatment = train[TREATMENT_COL].to_numpy(dtype=np.int8)
print("tree matrix:", X_tree_train.shape, X_tree_test.shape)

## Uplift Tree Implementation

In [ ]:
def _clip_prob(p):
    return np.clip(p, 1e-5, 1 - 1e-5)


def bernoulli_divergence(p_t, p_c, criterion="kl"):
    p_t = _clip_prob(p_t)
    p_c = _clip_prob(p_c)
    if criterion == "kl":
        kl_tc = p_t * np.log(p_t / p_c) + (1 - p_t) * np.log((1 - p_t) / (1 - p_c))
        kl_ct = p_c * np.log(p_c / p_t) + (1 - p_c) * np.log((1 - p_c) / (1 - p_t))
        return float(0.5 * (kl_tc + kl_ct))
    if criterion == "entropy":
        m = 0.5 * (p_t + p_c)
        h = lambda p: -(p * np.log(_clip_prob(p)) + (1 - p) * np.log(_clip_prob(1 - p)))
        return float(h(m) - 0.5 * h(p_t) - 0.5 * h(p_c))
    if criterion == "ed":
        return float((p_t - p_c) ** 2)
    raise ValueError(criterion)


@dataclass
class UpliftNode:
    value: float
    feature: int = -1
    threshold: float = 0.0
    left: object = None
    right: object = None


class UpliftTree:
    def __init__(self, max_depth=5, min_samples_leaf=800, min_treatment_leaf=120, max_features=14, n_thresholds=8, criterion="kl", random_state=42, smoothing=8.0):
        self.max_depth = int(max_depth)
        self.min_samples_leaf = int(min_samples_leaf)
        self.min_treatment_leaf = int(min_treatment_leaf)
        self.max_features = int(max_features)
        self.n_thresholds = int(n_thresholds)
        self.criterion = criterion
        self.random_state = int(random_state)
        self.smoothing = float(smoothing)
        self.rng = np.random.default_rng(self.random_state)
        self.root = None

    def _leaf_value(self, y, t):
        tr = t == 1
        ct = t == 0
        p_t = (y[tr].sum() + self.smoothing * y.mean()) / (tr.sum() + self.smoothing) if tr.sum() else y.mean()
        p_c = (y[ct].sum() + self.smoothing * y.mean()) / (ct.sum() + self.smoothing) if ct.sum() else y.mean()
        return float(p_t - p_c)

    def _node_score(self, y, t):
        tr = t == 1
        ct = t == 0
        if tr.sum() < self.min_treatment_leaf or ct.sum() < self.min_treatment_leaf:
            return 0.0
        p_t = (y[tr].sum() + self.smoothing * y.mean()) / (tr.sum() + self.smoothing)
        p_c = (y[ct].sum() + self.smoothing * y.mean()) / (ct.sum() + self.smoothing)
        return bernoulli_divergence(p_t, p_c, self.criterion)

    def _valid_leaf(self, y, t):
        if len(y) < self.min_samples_leaf:
            return False
        return ((t == 1).sum() >= self.min_treatment_leaf) and ((t == 0).sum() >= self.min_treatment_leaf)

    def _best_split(self, X, y, t):
        n, p = X.shape
        n_features = min(self.max_features, p)
        feat_idx = self.rng.choice(p, size=n_features, replace=False)
        parent_score = self._node_score(y, t)
        best_gain = 0.0
        best_feature = -1
        best_threshold = None

        for j in feat_idx:
            x = X[:, j]
            qs = np.linspace(0.08, 0.92, self.n_thresholds)
            thresholds = np.unique(np.quantile(x, qs))
            if len(thresholds) == 0:
                continue
            for thr in thresholds:
                left = x <= thr
                right = ~left
                if not self._valid_leaf(y[left], t[left]) or not self._valid_leaf(y[right], t[right]):
                    continue
                left_score = self._node_score(y[left], t[left])
                right_score = self._node_score(y[right], t[right])
                gain = (left.mean() * left_score + right.mean() * right_score) - parent_score
                if gain > best_gain:
                    best_gain = gain
                    best_feature = int(j)
                    best_threshold = float(thr)

        return best_feature, best_threshold, best_gain

    def _build(self, X, y, t, depth):
        node = UpliftNode(value=self._leaf_value(y, t))
        if depth >= self.max_depth or not self._valid_leaf(y, t):
            return node
        feature, threshold, gain = self._best_split(X, y, t)
        if feature < 0 or gain <= 1e-10:
            return node
        mask = X[:, feature] <= threshold
        node.feature = feature
        node.threshold = threshold
        node.left = self._build(X[mask], y[mask], t[mask], depth + 1)
        node.right = self._build(X[~mask], y[~mask], t[~mask], depth + 1)
        return node

    def fit(self, X, y, t):
        self.root = self._build(X, y.astype(np.float32), t.astype(np.int8), 0)
        return self

    def _predict_row(self, x, node):
        while node.feature >= 0:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node.value

    def predict(self, X):
        return np.array([self._predict_row(row, self.root) for row in X], dtype=np.float32)


class UpliftRandomForest:
    def __init__(self, n_estimators=80, max_samples=0.75, random_state=42, **tree_params):
        self.n_estimators = int(n_estimators)
        self.max_samples = float(max_samples)
        self.random_state = int(random_state)
        self.tree_params = dict(tree_params)
        self.trees = []

    def fit(self, X, y, t):
        rng = np.random.default_rng(self.random_state)
        n = len(y)
        sample_size = max(1000, int(n * self.max_samples))
        self.trees = []
        for i in range(self.n_estimators):
            idx = rng.choice(n, size=sample_size, replace=True)
            params = dict(self.tree_params)
            params["random_state"] = self.random_state + i * 17
            tree = UpliftTree(**params).fit(X[idx], y[idx], t[idx])
            self.trees.append(tree)
        return self

    def predict(self, X):
        preds = np.vstack([tree.predict(X) for tree in self.trees])
        return preds.mean(axis=0)

## Amount Model

In [ ]:
def categorical_features(df, features):
    return [c for c in features if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c])]


def prepare_catboost_X(df, features):
    X = df[features].copy()
    for c in categorical_features(X, features):
        X[c] = X[c].astype("string").fillna("__MISSING__")
    return X


def fit_amount_model(train_part, features, seed=RANDOM_STATE):
    pos = train_part[TARGET_COL] > 0
    X = prepare_catboost_X(train_part.loc[pos], features)
    y = np.log1p(train_part.loc[pos, TARGET_COL].to_numpy(dtype=float))
    model = CatBoostRegressor(
        iterations=900 if RUN_MODE == "smoke" else 1800,
        learning_rate=0.045,
        depth=5,
        l2_leaf_reg=55,
        random_strength=6,
        rsm=0.85,
        bagging_temperature=2,
        min_data_in_leaf=120,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=seed,
        allow_writing_files=False,
        thread_count=-1,
        verbose=False,
    )
    model.fit(X, y, cat_features=categorical_features(X, features))
    return model


def predict_amount(model, df, features):
    X = prepare_catboost_X(df, features)
    amount = np.expm1(model.predict(X))
    cap = np.nanpercentile(train[TARGET_COL].to_numpy(dtype=float), 99.9)
    return np.clip(amount, 0, cap)

## Validation Splits

In [ ]:
def make_strata(df):
    return df[TREATMENT_COL].astype(str) + "__" + df[COMM_COL].astype(str)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
strat_train_idx, strat_val_idx = next(sss.split(train, make_strata(train)))

id_cutoff = train[ID_COL].quantile(0.75)
stress_val_mask = train[ID_COL] >= id_cutoff
stress_train_idx = np.where(~stress_val_mask)[0]
stress_val_idx = np.where(stress_val_mask)[0]

SPLITS = {
    "strat": (strat_train_idx, strat_val_idx),
    "stress_user_id": (stress_train_idx, stress_val_idx),
}

for name, (_, val_idx) in SPLITS.items():
    print(name, len(val_idx))
    display(train.iloc[val_idx].groupby([COMM_COL, TREATMENT_COL])[TARGET_COL].agg(["size", "mean"]))

## Search

In [ ]:
def sample_config(rng, i):
    return {
        "config_id": i,
        "criterion": str(rng.choice(["kl", "entropy", "ed"])),
        "max_depth": int(rng.choice([CFG["max_depth"], max(3, CFG["max_depth"] - 1)])),
        "min_samples_leaf": int(rng.choice([600, 900, 1300, 1800, 2600])),
        "min_treatment_leaf": int(rng.choice([90, 140, 220, 320])),
        "max_features": int(rng.choice([10, 14, 18, 24])),
        "n_thresholds": int(CFG["n_thresholds"]),
        "smoothing": float(rng.choice([4.0, 8.0, 16.0, 32.0])),
        "max_samples": float(rng.choice([0.55, 0.70, 0.85])),
        "n_estimators": int(CFG["n_estimators"]),
    }


def fit_eval_config(config, split_name="strat", seed=RANDOM_STATE):
    tr_idx, va_idx = SPLITS[split_name]
    X_tr = X_tree_train[tr_idx]
    X_va = X_tree_train[va_idx]
    y_tr = y_purchase[tr_idx]
    t_tr = treatment[tr_idx]
    val_part = train.iloc[va_idx].reset_index(drop=True)

    forest = UpliftRandomForest(
        n_estimators=config["n_estimators"],
        max_samples=config["max_samples"],
        random_state=seed,
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"],
        min_treatment_leaf=config["min_treatment_leaf"],
        max_features=config["max_features"],
        n_thresholds=config["n_thresholds"],
        criterion=config["criterion"],
        smoothing=config["smoothing"],
    )
    forest.fit(X_tr, y_tr, t_tr)
    purchase_uplift = forest.predict(X_va)

    amount_model = fit_amount_model(train.iloc[tr_idx].reset_index(drop=True), FEATURES, seed=seed + 1000)
    amount = predict_amount(amount_model, val_part, FEATURES)
    spend_score = purchase_uplift * amount

    raw_metrics = evaluate_scores(val_part, purchase_uplift, f"uplift_forest_purchase_cfg{config['config_id']}_{split_name}")
    spend_metrics = evaluate_scores(val_part, spend_score, f"uplift_forest_spend_cfg{config['config_id']}_{split_name}")
    return raw_metrics, spend_metrics

rng = np.random.default_rng(RANDOM_STATE)
configs = [sample_config(rng, i) for i in range(CFG["n_configs"])]
rows = []

for config in configs:
    print("fit", config)
    start = time.time()
    raw, spend = fit_eval_config(config, split_name="strat", seed=RANDOM_STATE + config["config_id"] * 19)
    for kind, metrics in [("purchase", raw), ("spend", spend)]:
        metrics.update({
            "score_kind": kind,
            "split": "strat",
            "elapsed_min": (time.time() - start) / 60,
            **{f"param_{k}": v for k, v in config.items()},
        })
        metrics["objective"] = objective(metrics)
        rows.append(metrics)
    results = pd.DataFrame(rows).sort_values("objective", ascending=False)
    results.to_csv(ARTIFACT_DIR / "uplift_tree_search_results.csv", index=False)
    display(results.head(10))
    gc.collect()

results = pd.DataFrame(rows).sort_values("objective", ascending=False).reset_index(drop=True)
display(results.head(20))

## Stress Recheck

In [ ]:
stress_rows = []
best_cfg_rows = results.drop_duplicates(["param_config_id", "score_kind"]).head(min(5, len(results)))

for _, row in best_cfg_rows.iterrows():
    config = {k.replace("param_", ""): row[k] for k in row.index if k.startswith("param_")}
    for k in ["config_id", "max_depth", "min_samples_leaf", "min_treatment_leaf", "max_features", "n_thresholds", "n_estimators"]:
        config[k] = int(config[k])
    for k in ["smoothing", "max_samples"]:
        config[k] = float(config[k])
    print("stress", config)
    raw, spend = fit_eval_config(config, split_name="stress_user_id", seed=RANDOM_STATE + config["config_id"] * 31)
    picked = spend if row["score_kind"] == "spend" else raw
    picked.update({
        "score_kind": row["score_kind"],
        "split": "stress_user_id",
        **{f"param_{k}": v for k, v in config.items()},
    })
    picked["stress_objective"] = objective(picked)
    picked["strat_objective"] = float(row["objective"])
    stress_rows.append(picked)
    stress_df = pd.DataFrame(stress_rows).sort_values("stress_objective", ascending=False)
    stress_df.to_csv(ARTIFACT_DIR / "uplift_tree_stress_results.csv", index=False)
    display(stress_df)

stress_df = pd.DataFrame(stress_rows)
selected = results.merge(
    stress_df[["param_config_id", "score_kind", "stress_objective", "lower80", "uplift_at10"]].rename(columns={"lower80": "stress_lower80", "uplift_at10": "stress_uplift_at10"}),
    on=["param_config_id", "score_kind"],
    how="left",
)
selected["combined_objective"] = (
    0.58 * selected["objective"]
    + 0.30 * selected["stress_objective"].fillna(selected["objective"])
    + 0.12 * np.minimum(selected["lower80"], selected["stress_lower80"].fillna(selected["lower80"]))
)
selected = selected.sort_values("combined_objective", ascending=False).reset_index(drop=True)
selected.to_csv(ARTIFACT_DIR / "uplift_tree_selected_results.csv", index=False)
display(selected.head(15))

## Final Training And Submissions

In [ ]:
def config_from_row(row):
    config = {k.replace("param_", ""): row[k] for k in row.index if k.startswith("param_")}
    for k in ["config_id", "max_depth", "min_samples_leaf", "min_treatment_leaf", "max_features", "n_thresholds", "n_estimators"]:
        config[k] = int(config[k])
    for k in ["smoothing", "max_samples"]:
        config[k] = float(config[k])
    config["criterion"] = str(config["criterion"])
    return config

best = selected.iloc[0]
best_config = config_from_row(best)
print("best config:", json.dumps(best_config, indent=2))

seed_scores = []
seed_purchase_scores = []
for seed in CFG["final_seeds"]:
    print("final seed", seed)
    forest = UpliftRandomForest(
        n_estimators=best_config["n_estimators"],
        max_samples=best_config["max_samples"],
        random_state=seed,
        max_depth=best_config["max_depth"],
        min_samples_leaf=best_config["min_samples_leaf"],
        min_treatment_leaf=best_config["min_treatment_leaf"],
        max_features=best_config["max_features"],
        n_thresholds=best_config["n_thresholds"],
        criterion=best_config["criterion"],
        smoothing=best_config["smoothing"],
    )
    forest.fit(X_tree_train, y_purchase, treatment)
    purchase_uplift_test = forest.predict(X_tree_test)
    seed_purchase_scores.append(purchase_uplift_test)

    amount_model = fit_amount_model(train, FEATURES, seed=seed + 5000)
    amount_test = predict_amount(amount_model, test, FEATURES)
    seed_scores.append(purchase_uplift_test * amount_test)
    gc.collect()

purchase_ensemble = rank_average(*seed_purchase_scores)
spend_ensemble = rank_average(*seed_scores)

save_submission(purchase_ensemble, "predictions_uplift_tree_purchase.csv")
save_submission(spend_ensemble, "predictions_uplift_tree_spend.csv")

# Optional blend with an existing CatBoost predictions.csv. Use only if that file is your current best trusted CatBoost candidate.
base_path = Path("predictions.csv")
if base_path.exists():
    base = pd.read_csv(base_path)
    if len(base) == len(test) and base[ID_COL].equals(test[ID_COL].reset_index(drop=True)):
        blend_80_20 = rank_average(base[base.columns[-1]].to_numpy(), spend_ensemble, weights=[0.80, 0.20])
        blend_70_30 = rank_average(base[base.columns[-1]].to_numpy(), spend_ensemble, weights=[0.70, 0.30])
        save_submission(blend_80_20, "predictions_uplift_tree_blend_existing_80_20.csv")
        save_submission(blend_70_30, "predictions_uplift_tree_blend_existing_70_30.csv")

summary = []
for name, scores in {
    "uplift_tree_purchase": purchase_ensemble,
    "uplift_tree_spend": spend_ensemble,
}.items():
    top_idx = np.argsort(scores)[-int(len(scores) * TOP_K):]
    top = test.iloc[top_idx]
    row = {"candidate": name, "score_mean": float(np.mean(scores)), "score_std": float(np.std(scores))}
    for comm, share in top[COMM_COL].value_counts(normalize=True).sort_index().items():
        row[f"top10_{comm}_share"] = float(share)
    summary.append(row)
summary_df = pd.DataFrame(summary)
summary_df.to_csv(ARTIFACT_DIR / "uplift_tree_submission_summary.csv", index=False)
display(summary_df)

## Submit Order

Если попыток мало, я бы отправлял в таком порядке:

1. `predictions_uplift_tree_blend_existing_80_20.csv`, если в папке был сильный CatBoost `predictions.csv`;
2. `predictions_uplift_tree_spend.csv`, если локальный `lower80` не хуже текущего CatBoost;
3. `predictions_uplift_tree_purchase.csv` только как разведку, потому что он ранжирует вероятность инкрементальной покупки, а не spend uplift.